# The knowledge graph (gnd_context), step by step

This notebook shows how `ie-agent build-gnd` builds the knowledge graph, what the graph looks like and how complete it is.

**How the graph is put together:**

| Step | Source | Result | Content |
|---|---|---|---|
| 1 | `vocab/GND-Subjects-plus.ttl` (DVC) | `gnd_graph/graph.pkl` | terms and their relations |
| 2 | DNB dumps of subject headings and places | `gnd_graph/details.pkl` | definition, description, using instructions, subject category, kind of relation |
| 3 | headings of the training documents | computed at run time | how often a term was assigned |

The notebook **only reads**: it repeats the steps in memory and writes nothing to `gnd_graph/`. It needs neither a GPU
nor Weaviate, only the built files in `gnd_graph/`. All numbers are computed when the notebook runs.

In [1]:
import gzip
import json
import pickle
from collections import Counter, defaultdict
from html import escape

import pandas as pd
import yaml
from IPython.display import HTML, Markdown, display

from ie_agent.config import CONFIG, path
from ie_agent.data import load_gold
from ie_agent.gnd import Gnd, gnd_context
from ie_agent.gnd.build import _NAME, _parse_details, _parse_graph, gnd_files, nfc

EXAMPLE = "Klimaschutz"              # a term with a broader, a narrower and related terms
EXAMPLE_DEFINITION = "Wissensgraph"  # a term that has a definition
RUN = "reasoning-off_gnd_context-on_toc-off_examples-optional"   # the row "checked by the agent" comes from this run

cfg = yaml.safe_load(CONFIG.read_text())
files = gnd_files(cfg)


def percent(part, whole):
    return f"{part / whole:.1%}" if whole else "–"


def number(n):
    return f"{n:,}"


def bars(rows, whole, label="share"):
    """A small bar table from (name, count), without additional packages."""
    html = ["<table>", f"<tr><th style='text-align:left'></th><th>count</th><th>{label}</th><th></th></tr>"]
    for name, n in rows:
        width = 260 * n / whole if whole else 0
        html.append(f"<tr><td style='text-align:left'>{escape(str(name))}</td><td>{number(n)}</td>"
                    f"<td>{percent(n, whole)}</td><td style='text-align:left;min-width:270px'>"
                    f"<div style='background:#4c78a8;height:12px;width:{width:.0f}px'></div></td></tr>")
    display(HTML("".join(html) + "</table>"))

## 1. The sources

While building, `manifest.json` records exactly which files were used, so a run can later be traced back to the exact
GND version.

In [2]:
manifest = json.loads(files["manifest"].read_text())
print("built:", manifest["built"])
display(pd.DataFrame(manifest["sources"])[["file", "bytes", "last_modified", "note"]].fillna(""))
print({k: v for k, v in manifest.items() if k not in ("built", "sources")})

built: 2026-10-30T17:09:35


,file,bytes,last_modified,note
0,authorities-gnd-sachbegriff_lds.ttl.gz,24119388,"Mon, 02 Mar 2026 13:19:09 GMT",
1,authorities-gnd-geografikum_lds.ttl.gz,44347076,"Mon, 02 Mar 2026 13:20:00 GMT",
2,gnd-sc.ttl,137454,,
3,/home/harbc/ki-fsprompt/vocab/GND-Subjects-plu...,70495107,,GND vocabulary of the pipeline (DVC)


{'concepts': 309417, 'with_details': 197252, 'with_definition': 47127, 'with_description': 14449, 'with_using_instructions': 10951}


## 2. Step 1: the vocabulary as a graph

### 2.1 The raw file

`GND-Subjects-plus.ttl` is written in SKOS. Each term is one block, with a blank line in between:

| Statement | Meaning |
|---|---|
| `skos:prefLabel` | preferred name of the heading |
| `skos:altLabel`, `skos:hiddenLabel` | other names (synonyms, spellings) |
| `skos:broader` | broader term |
| `skos:related` | related term |

**Narrower terms are not in the file.** They are derived from the broader terms in 2.2. The number after `gnd:` is the
**IDN**, not the GND number, which matters in step 2.

In [3]:
ttl = path(cfg, "gnd_ttl")
text = ttl.read_text(encoding="utf-8")
blocks = text.split("\n\n")

example_block = next(b for b in blocks if f'skos:prefLabel "{EXAMPLE}"@de' in b)
print(example_block)

gnd:981714250 a skos:Concept;
  skos:broader gnd:040616444;
  skos:altLabel "Klima / Umweltpolitik"@de,
    "Klimaänderung / Umweltschutz"@de,
    "Klima / Umweltschutz"@de,
    "Klimaschutzpolitik"@de,
    "Klimapolitik"@de;
  skos:prefLabel "Klimaschutz"@de.


### 2.2 From text to graph

`_parse_graph` goes through all blocks and creates several dictionaries. It computes two things beyond the file:

- **`narrower`:** the reverse of `broader`. If A is a broader term of B, then B is a narrower term of A.
- **`related` (both directions):** the file often lists a relation on one side only, so it is made symmetric.
  `related_out` keeps only what is in the term's own record. `gnd_context` shows the model only this direction, because
  reverse links often come from persons and works.

The next cell rebuilds the graph and checks that it is identical to `graph.pkl`.

In [4]:
g = _parse_graph(ttl)
stored = pickle.loads(files["graph"].read_bytes())
print("identical to graph.pkl:", g == stored)

example_id = next(i for i, name in g["pref"].items() if name == EXAMPLE)


def names(ids):
    return sorted(g["pref"].get(i, f"outside the vocabulary ({i})") for i in ids)


pd.DataFrame([
    ("pref", "IDN -> name", number(len(g["pref"])), g["pref"][example_id]),
    ("alt", "IDN -> number of other names", number(len(g["alt"])), g["alt"].get(example_id, 0)),
    ("broader", "IDN -> broader terms", number(len(g["broader"])), names(g["broader"].get(example_id, ()))),
    ("narrower", "IDN -> narrower terms (derived)", number(len(g["narrower"])), names(g["narrower"].get(example_id, ()))),
    ("related_out", "IDN -> related, own record", number(len(g["related_out"])), names(g["related_out"].get(example_id, ()))),
    ("related", "IDN -> related, both directions", number(len(g["related"])), names(g["related"].get(example_id, ()))),
], columns=["dictionary", "content", "entries", f"example: {EXAMPLE}"])

identical to graph.pkl: True


,dictionary,content,entries,example: Klimaschutz
0,pref,IDN -> name,"309,417",Klimaschutz
1,alt,IDN -> number of other names,"193,864",5
2,broader,IDN -> broader terms,"193,860",[Umweltschutz]
3,narrower,IDN -> narrower terms (derived),"40,014",[Sequestrierung]
4,related_out,"IDN -> related, own record","140,294",[]
5,related,"IDN -> related, both directions","215,587","[Bundespreis Blauer Kompass, COP (17, 2011, Du..."


### 2.3 The graph around one term

Broader terms are at the top, narrower terms at the bottom, related terms at the sides. Change `EXAMPLE` above for a
different picture.

In [5]:
def draw(gnd_id, limit=8):
    """The direct neighbourhood of a term as a small SVG."""
    width, height, cx, cy = 900, 430, 450, 215
    colors = {"broader": "#4c78a8", "narrower": "#54a24b", "related": "#b279a2"}
    nodes = []

    def spread(ids, y, kind):
        ids = sorted(ids, key=lambda i: g["pref"].get(i, i))[:limit]
        for n, i in enumerate(ids):
            nodes.append(((n + 1) * width / (len(ids) + 1), y + (n % 2) * 26, i, kind))

    spread(g["broader"].get(gnd_id, ()), 40, "broader")
    spread(g["narrower"].get(gnd_id, ()), 365, "narrower")
    related = sorted(g["related"].get(gnd_id, ()), key=lambda i: g["pref"].get(i, i))[:limit]
    for n, i in enumerate(related):
        side = 1 if n % 2 else -1
        row = n // 2
        nodes.append((cx + side * (220 + 40 * (row % 2)), 135 + row * 50, i, "related"))

    svg = [f"<svg width='{width}' height='{height}' xmlns='http://www.w3.org/2000/svg' font-family='sans-serif' font-size='12'>"]
    for x, y, _, kind in nodes:
        dash = " stroke-dasharray='4 3'" if kind == "related" else ""
        svg.append(f"<line x1='{cx}' y1='{cy}' x2='{x:.0f}' y2='{y:.0f}' stroke='{colors[kind]}'{dash}/>")
    for x, y, i, kind in nodes:
        label = g["pref"].get(i, i)
        label = label if len(label) <= 26 else label[:25] + "…"
        box = 7 * len(label) + 12
        svg.append(f"<rect x='{x - box / 2:.0f}' y='{y - 11:.0f}' width='{box}' height='22' rx='4' fill='white' stroke='{colors[kind]}'/>"
                   f"<text x='{x:.0f}' y='{y + 4:.0f}' text-anchor='middle' fill='#222'>{escape(label)}</text>")
    label = g["pref"][gnd_id]
    box = 8 * len(label) + 20
    svg.append(f"<rect x='{cx - box / 2}' y='{cy - 15}' width='{box}' height='30' rx='5' fill='#f58518'/>"
               f"<text x='{cx}' y='{cy + 5}' text-anchor='middle' fill='white' font-weight='bold'>{escape(label)}</text>")
    svg.append("</svg>")
    display(HTML("".join(svg)))
    counts = [(kind, len(g[kind].get(gnd_id, ()))) for kind in ("broader", "narrower", "related")]
    print("in total:", dict(counts), f"(at most {limit} of each kind are drawn)")
    print("blue = broader, green = narrower, dashed purple = related")


draw(example_id)

in total: {'broader': 1, 'narrower': 1, 'related': 10} (at most 8 of each kind are drawn)
blue = broader, green = narrower, dashed purple = related


## 3. Step 2: the details from the DNB dumps

### 3.1 A dump record

The dumps contain what the vocabulary lacks: `gndo:definition`, `gndo:biographicalOrHistoricalInformation`,
`gndo:usingInstructions` and `gndo:gndSubjectCategory`. They also name the **kind of relation** to the broader term:
`broaderTermGeneric` (kind of), `broaderTermPartitive` (part of), `broaderTermInstantial` (instance of). The unspecific
`broaderTermGeneral` does not count as a kind.

In [6]:
dump_records = defaultdict(list)
target_number = pickle.loads(files["details"].read_bytes())[example_id]["gnd_nr"]
with gzip.open(files["dumps"][0], "rt", encoding="utf-8") as f:
    active = False
    for line in f:
        if line.startswith("<https://d-nb.info/gnd/"):
            active = line.startswith(f"<https://d-nb.info/gnd/{target_number}>") or (
                active and line.startswith(f"<https://d-nb.info/gnd/{target_number}/"))
        if active:
            dump_records[target_number].append(line.rstrip())
print(f"{EXAMPLE}: IDN {example_id}, GND number {target_number}\n")
print("\n".join(dump_records[target_number]))

Klimaschutz: IDN 981714250, GND number 7547705-1

<https://d-nb.info/gnd/7547705-1> a gndo:SubjectHeadingSensoStricto;
  wdrs:describedby <https://d-nb.info/gnd/7547705-1/about> .

<https://d-nb.info/gnd/7547705-1/about> dcterms:license <http://creativecommons.org/publicdomain/zero/1.0/>;
  dcterms:modified "2025-10-23T13:08:53.000"^^xsd:dateTime;
  gndo:descriptionLevel <https://d-nb.info/standards/vocab/gnd/description-level#1> .

<https://d-nb.info/gnd/7547705-1> skos:exactMatch <http://zbw.eu/stw/descriptor/19481-5>,
    <http://zbw.eu/stw/descriptor/29577-0>;
  skos:narrowMatch <http://zbw.eu/stw/descriptor/30306-0>;
  gndo:gndIdentifier "7547705-1";
  owl:sameAs <http://www.wikidata.org/entity/Q898653>;
  gndo:oldAuthorityNumber "(DE-588c)7547705-1";
  gndo:relatedDdcWithDegreeOfDeterminacy1 <http://dewey.info/class/333.72/>;
  gndo:relatedDdcWithDegreeOfDeterminacy2 <http://dewey.info/class/363.73874/>;
  gndo:variantNameForTheSubjectHeading "Klimaschutzpolitik", "Klima / Umwelt

### 3.2 The problem: two different numbers

The vocabulary identifies a term by its **IDN**, the dumps by its **GND number**. The two sources share no number.
`_parse_details` therefore connects them through the **name**, but only if it is unambiguous: the name must occur exactly
once in the vocabulary and exactly once in the loaded dumps. No entry is better than the details of a wrong term.

The next cell rebuilds the details and checks that they are identical to `details.pkl`. This takes about a minute.

In [7]:
details = _parse_details(g, files["dumps"], files["classification"])
print("identical to details.pkl:", details == pickle.loads(files["details"].read_bytes()))

for i in (example_id, next(k for k, n in g["pref"].items() if n == EXAMPLE_DEFINITION)):
    print(f"\n{g['pref'][i]} (IDN {i})")
    print(json.dumps(details[i], ensure_ascii=False, indent=2))

identical to details.pkl: True

Klimaschutz (IDN 981714250)
{
  "gnd_nr": "7547705-1",
  "definition": null,
  "description": null,
  "using_instructions": null,
  "classification": [
    "Umweltschutz, Umweltbelastung",
    "Meteorologie, Klimatologie, Hochatmosphäre, Magnetosphäre"
  ],
  "broader_types": {
    "040616444": "Generic"
  }
}

Wissensgraph (IDN 1241153396)
{
  "gnd_nr": "1241153396",
  "definition": "Ein Wissensgraph ist eine Wissensbasis, die ein graphisch stukturiertes Datenmodell oder eine Topologie zur Integration von Daten verwendet.",
  "description": null,
  "using_instructions": null,
  "classification": [
    "Informatik, Datenverarbeitung",
    "Bibliothek, Information und Dokumentation"
  ],
  "broader_types": {}
}


### 3.3 Why do some terms have no details?

This cell puts every term of the vocabulary into exactly one group. The largest group shows where an improvement would
have to start. A name may also just be written differently, for example `(Motiv)` instead of `<Motiv>`, so "name not in
the loaded dumps" does not always mean the term is really missing there.

In [8]:
names_in_dumps = Counter()
for dump in files["dumps"]:
    with gzip.open(dump, "rt", encoding="utf-8") as f:
        for line in f:
            m = _NAME.search(line)
            if m:
                names_in_dumps[nfc(m.group(1))] += 1
names_in_vocab = Counter(nfc(name) for name in g["pref"].values())

reasons = Counter()
for i, name in g["pref"].items():
    name = nfc(name)
    if i in details:
        reasons["details available"] += 1
    elif names_in_dumps[name] == 0:
        reasons["name not in the loaded dumps"] += 1
    elif names_in_vocab[name] > 1:
        reasons["name occurs several times in the vocabulary"] += 1
    else:
        reasons["name occurs several times in the dumps"] += 1
bars(reasons.most_common(), len(g["pref"]))

,count,share,
details available,"197,252",63.7%,
name not in the loaded dumps,"111,719",36.1%,
name occurs several times in the dumps,386,0.1%,
name occurs several times in the vocabulary,60,0.0%,


## 4. Step 3: the frequency from training

At start-up, `Gnd.load` counts how often each term was assigned in the **training documents**. This later tells the model
whether a term is used often or rarely. Validate documents are not counted, so no answers from the evaluation data reach
the agent.

In [9]:
gnd = Gnd.load(cfg)
assert gnd.g == g and gnd.details == details
used = [i for i in g["pref"] if gnd.frequency.get(i, 0) > 0]
print("assignments in training:", number(sum(gnd.frequency.get(i, 0) for i in g["pref"])))
print("terms assigned at least once in training:", number(len(used)), f"({percent(len(used), len(g['pref']))})")
print()
display(pd.DataFrame([(g["pref"][i], gnd.frequency[i], len(g["narrower"].get(i, ())))
                      for i in sorted(used, key=lambda i: -gnd.frequency[i])[:10]],
                     columns=["most frequent terms", "assignments", "narrower terms"]))

assignments in training: 47,827
terms assigned at least once in training: 17,302 (5.6%)



,most frequent terms,assignments,narrower terms
0,Deutschland,549,3
1,Tiermodell,289,0
2,Finite-Elemente-Methode,152,7
3,Prüfstand,133,6
4,Hund,123,33
5,Produktentwicklung,107,10
6,Simulation,105,26
7,Herstellung,103,5
8,Bemessung,99,2
9,Pferd,95,23


## 5. What the agent finally sees

The agent never reads the graph directly. It calls `gnd_context` and gets this answer for each candidate; the field names
are German because the model reads them. Narrower terms are limited to 8. If there are more, the real agent picks the ones
closest to the document through Weaviate. Weaviate is not used here, so `NoWeaviate` replaces this one step and the list is
filled by frequency. **The selection of narrower terms can therefore differ in a real run.** The sentence in `state` is
made up and only serves the `im_text` marker.

In [10]:
class NoWeaviate:
    def closest(self, text, ids, top):
        return []


state = {"text": ["Die Arbeit untersucht Klimaschutz und Umweltschutz in Kommunen."],
         "mapped": {example_id: EXAMPLE}, "graph": {}}
answer = gnd_context(gnd, example_id, state, NoWeaviate(), cfg["agent"]["max_narrower"])
print(json.dumps(answer, ensure_ascii=False, indent=2))
print("\nIDs the agent may additionally assign in finish() after this:", number(len(state["graph"])))

{
  "gnd_id": "981714250",
  "begriff": "Klimaschutz",
  "definition": null,
  "sachgruppen": [
    "Umweltschutz, Umweltbelastung",
    "Meteorologie, Klimatologie, Hochatmosphäre, Magnetosphäre"
  ],
  "in_trainingsdokumenten_vergeben": 13,
  "oberbegriffe": [
    {
      "gnd_id": "040616444",
      "begriff": "Umweltschutz",
      "beziehung": "Kandidat ist eine Art davon"
    }
  ],
  "unterbegriffe": {
    "anzahl": 1,
    "auswahl": [
      {
        "gnd_id": "98845436X",
        "begriff": "Sequestrierung"
      }
    ]
  },
  "verwandte_begriffe": [],
  "beziehungen_zu_deinen_kandidaten": []
}

IDs the agent may additionally assign in finish() after this: 2


## 6. How complete is the graph?

### 6.1 Broader, narrower and related terms

One might assume that nearly all terms have broader and narrower terms. This cell measures it for two groups: all terms
of the vocabulary, and only the terms actually assigned in the 827 validate documents.

In [11]:
docs_validate = pd.read_csv(path(cfg, "sections") / "abstract_intro/validate.csv", dtype=str)["doc_id"].tolist()
gold = load_gold(cfg)
validate_tokens = [i for doc in docs_validate for i in gold[doc]]
validate_terms = set(validate_tokens)
vocab = set(g["pref"])


def has(i, kind):
    return bool(g[kind].get(i))


checks = {
    "has a broader term": lambda i: has(i, "broader"),
    "has a narrower term": lambda i: has(i, "narrower"),
    "has broader AND narrower": lambda i: has(i, "broader") and has(i, "narrower"),
    "has broader OR narrower": lambda i: has(i, "broader") or has(i, "narrower"),
    "has a related term": lambda i: has(i, "related"),
    "has no relation at all": lambda i: not (has(i, "broader") or has(i, "narrower") or has(i, "related")),
}
inside = [i for i in validate_terms if i in vocab]
coverage = pd.DataFrame([(name, number(sum(map(test, vocab))), percent(sum(map(test, vocab)), len(vocab)),
                          percent(sum(map(test, inside)), len(inside)))
                         for name, test in checks.items()],
                        columns=["relation", "terms in the vocabulary", "share of the vocabulary",
                                 "share of terms assigned in validate"])
display(coverage)
bars([(name, sum(map(test, vocab))) for name, test in checks.items()], len(vocab))
print("broader links pointing outside the vocabulary:",
      number(sum(1 for i in vocab for b in g["broader"].get(i, ()) if b not in vocab)))

,relation,terms in the vocabulary,share of the vocabulary,share of terms assigned in validate
0,has a broader term,"193,860",62.7%,66.5%
1,has a narrower term,"39,216",12.7%,58.2%
2,has broader AND narrower,"27,105",8.8%,37.3%
3,has broader OR narrower,"205,971",66.6%,87.4%
4,has a related term,"150,991",48.8%,56.6%
5,has no relation at all,"36,013",11.6%,4.6%


,count,share,
has a broader term,"193,860",62.7%,
has a narrower term,"39,216",12.7%,
has broader AND narrower,"27,105",8.8%,
has broader OR narrower,"205,971",66.6%,
has a related term,"150,991",48.8%,
has no relation at all,"36,013",11.6%,


broader links pointing outside the vocabulary: 1,032


### 6.2 How many narrower terms does a term have?

This matters for the limit of 8 narrower terms: only where there are more than 8 can the selection change anything.

In [12]:
groups = Counter()
for i in vocab:
    n = len(g["narrower"].get(i, ()))
    groups["0" if n == 0 else "1" if n == 1 else "2 to 8" if n <= 8 else "9 to 50" if n <= 50 else "more than 50"] += 1
bars([(k, groups[k]) for k in ("0", "1", "2 to 8", "9 to 50", "more than 50")], len(vocab))
top = sorted(vocab, key=lambda i: -len(g["narrower"].get(i, ())))[:8]
display(pd.DataFrame([(g["pref"][i], len(g["narrower"].get(i, ()))) for i in top],
                     columns=["most narrower terms", "count"]))

,count,share,
0,"270,201",87.3%,
1,"16,908",5.5%,
2 to 8,"17,761",5.7%,
9 to 50,"4,095",1.3%,
more than 50,452,0.1%,


,most narrower terms,count
0,Motiv,11169
1,Programm,3809
2,Mundart,2505
3,Familie,1195
4,Adel,851
5,Kirchenbau,788
6,Schlacht,736
7,Preis (Auszeichnung),680


### 6.3 Which details are available?

`broader_types` counts the terms for which at least one kind of relation is known. The second table counts the individual
relations.

In [13]:
fields = {"any details": lambda d: True, "subject category": lambda d: d["classification"],
          "kind of broader relation": lambda d: d["broader_types"], "definition": lambda d: d["definition"],
          "description": lambda d: d["description"], "using instructions": lambda d: d["using_instructions"]}
bars([(name, sum(1 for d in details.values() if test(d))) for name, test in fields.items()], len(vocab),
     label="share of the vocabulary")
kinds = Counter(k for d in details.values() for k in d["broader_types"].values())
words = {"Generic": "kind of", "Partitive": "part of", "Instantial": "instance of"}
bars([(words[k], n) for k, n in kinds.most_common()], sum(kinds.values()), label="share of the relations")

,count,share of the vocabulary,
any details,"197,252",63.7%,
subject category,"181,772",58.7%,
kind of broader relation,"61,432",19.9%,
definition,"47,127",15.2%,
description,"14,449",4.7%,
using instructions,"10,951",3.5%,


,count,share of the relations,
kind of,"33,165",47.1%,
instance of,"27,462",39.0%,
part of,"9,790",13.9%,


### 6.4 Missing details: how much do they matter in practice?

Many terms without details are not automatically a big problem. What counts is how often exactly these terms are really
assigned or checked by the agent. Assignments are counted one by one, for the 827 validate documents only.

In [14]:
checked = []
with open(f"results/Qwen3.8-27B-FP8/Faktoren_analysis/{RUN}/log.jsonl") as f:
    for line in f:
        for step in json.loads(line)["steps"]:
            if step["tool"] == "gnd_context":
                checked += [i for i in (str(x).strip() for x in step["args"].get("gnd_ids", [])) if i in vocab]

known = set(details)
rows = [("terms in the vocabulary", len(vocab), len(vocab & known)),
        ("assignments in validate (827 documents)", len(validate_tokens), sum(i in known for i in validate_tokens)),
        ("terms checked by the agent (one run)", len(checked), sum(i in known for i in checked)),
        ("assignments in training", sum(gnd.frequency.get(i, 0) for i in vocab), sum(gnd.frequency.get(i, 0) for i in vocab & known))]
display(pd.DataFrame([(name, number(whole), number(part), percent(part, whole), percent(whole - part, whole))
                      for name, whole, part in rows],
                     columns=["group", "total", "with details", "share with details", "share without details"]))
outside = [i for i in validate_tokens if i not in vocab]
print("assignments in validate whose term is not in the vocabulary:", number(len(outside)),
      f"({percent(len(outside), len(validate_tokens))}). The agent can never hit these.")

,group,total,with details,share with details,share without details
0,terms in the vocabulary,"309,417","197,252",63.7%,36.3%
1,assignments in validate (827 documents),"4,018","3,652",90.9%,9.1%
2,terms checked by the agent (one run),"9,003","8,537",94.8%,5.2%
3,assignments in training,"47,827","45,584",95.3%,4.7%


assignments in validate whose term is not in the vocabulary: 43 (1.1%). The agent can never hit these.


## 7. Summary

The text below is generated from the measured numbers.

In [15]:
n = len(vocab)
broader = sum(has(i, "broader") for i in vocab)
narrower = sum(has(i, "narrower") for i in vocab)
both = sum(has(i, "broader") and has(i, "narrower") for i in vocab)
none = sum(not (has(i, "broader") or has(i, "narrower") or has(i, "related")) for i in vocab)
display(Markdown(f"""
- **Size:** {number(n)} terms, {percent(len(vocab & known), n)} of them have details.
- **Broader terms:** {percent(broader, n)} of the terms have one.
- **Narrower terms:** {percent(narrower, n)} of the terms have one.
- **Both:** {percent(both, n)}.
- **No relation at all:** {percent(none, n)}.
- **Details in practice:** {percent(rows[1][2], rows[1][1])} of the validate assignments and {percent(rows[2][2], rows[2][1])} of the checked terms have details.

The structure comes entirely from the vocabulary, so missing details change nothing about broader and narrower terms. A
term without a broader term only means that the vocabulary records none.
"""))


- **Size:** 309,417 terms, 63.7% of them have details.
- **Broader terms:** 62.7% of the terms have one.
- **Narrower terms:** 12.7% of the terms have one.
- **Both:** 8.8%.
- **No relation at all:** 11.6%.
- **Details in practice:** 90.9% of the validate assignments and 94.8% of the checked terms have details.

The structure comes entirely from the vocabulary, so missing details change nothing about broader and narrower terms. A
term without a broader term only means that the vocabulary records none.
